# Top-K span oracle report

Upload or locate `topk_span_oracle_artifacts.zip`. This notebook performs no training or inference and does not require the dataset.

## Step 1 — Load and validate the artifact


In [1]:
import json,csv,io,zipfile,sys,collections
from pathlib import Path
from IPython.display import display,Markdown
NAME="topk_span_oracle_artifacts.zip"
paths=[Path(NAME),Path("/content")/NAME,Path("/content/drive/MyDrive/CITS4012_A2_oracle")/NAME]
archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import drive; drive.mount("/content/drive"); archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import files
    up=files.upload(); assert NAME in up; archive=Path("/content")/NAME; archive.write_bytes(up[NAME])
assert archive is not None
zf=zipfile.ZipFile(archive); assert zf.testzip() is None
summary=json.loads(zf.read("topk_oracle_results/oracle_summary.json")); details=list(csv.DictReader(io.StringIO(zf.read("topk_oracle_results/oracle_details.csv").decode())))
print("Loaded",archive,"source",summary["source_run"],"N",summary["n"],"— no training or inference")


Loaded topk_span_oracle_artifacts.zip source CurrentGated-s1-20261014-175436 N 1076 — no training or inference


## Step 2 — Measure the Top-K oracle ceiling


In [2]:
lines=["| K | Oracle ROUGE-L | Oracle F1 | Oracle EM | Exact-answer coverage | ΔROUGE-L vs Top-1 |","|---:|---:|---:|---:|---:|---:|"]
for K in summary["ks"]:
    v=summary["ks"][K]; gain=v["rougeL"]-summary["ks"]["1"]["rougeL"]
    lines.append(f"| {K} | {v['rougeL']:.2f} | {v['f1']:.2f} | {v['em']:.2f} | {v['exact_coverage']:.2f}% | {gain:+.2f} |")
display(Markdown("## Top-K oracle ceiling\n\n"+"\n".join(lines)+"\n\nEach oracle metric independently selects the best candidate using gold answers; these are ceilings, not deployable scores."))


## Top-K oracle ceiling

| K | Oracle ROUGE-L | Oracle F1 | Oracle EM | Exact-answer coverage | ΔROUGE-L vs Top-1 |
|---:|---:|---:|---:|---:|---:|
| 1 | 44.14 | 44.22 | 27.32 | 27.32% | +0.00 |
| 3 | 56.61 | 56.60 | 38.94 | 38.94% | +12.47 |
| 5 | 61.57 | 61.57 | 43.31 | 43.31% | +17.43 |
| 10 | 67.16 | 67.30 | 49.16 | 49.16% | +23.02 |
| 20 | 70.99 | 71.36 | 53.81 | 53.81% | +26.85 |

Each oracle metric independently selects the best candidate using gold answers; these are ceilings, not deployable scores.

## Step 3 — Inspect question and region slices


In [3]:
for field,title in (("question_type","Question type"),("silver_region","Best silver-span region")):
    lines=[f"| {title} | N | Top-10 ΔROUGE-L | Top-10 ΔF1 | Top-10 ΔEM |","|---|---:|---:|---:|---:|"]
    for name,v in sorted(summary["slices_k10"][field].items()):
        lines.append(f"| {name} | {v['n']} | {v['rougeL']:+.2f} | {v['f1']:+.2f} | {v['em']:+.2f} |")
    display(Markdown(f"## {title} headroom\n\n"+"\n".join(lines)))


## Question type headroom

| Question type | N | Top-10 ΔROUGE-L | Top-10 ΔF1 | Top-10 ΔEM |
|---|---:|---:|---:|---:|
| how | 85 | +24.66 | +24.65 | +20.00 |
| other | 95 | +20.65 | +20.44 | +15.79 |
| what | 462 | +25.88 | +25.86 | +23.38 |
| when | 24 | +15.85 | +16.69 | +29.17 |
| where | 75 | +19.98 | +20.00 | +25.33 |
| which | 22 | +16.95 | +16.36 | +13.64 |
| who | 290 | +20.78 | +20.99 | +22.76 |
| why | 23 | +20.88 | +21.68 | +0.00 |

## Best silver-span region headroom

| Best silver-span region | N | Top-10 ΔROUGE-L | Top-10 ΔF1 | Top-10 ΔEM |
|---|---:|---:|---:|---:|
| body | 820 | +26.51 | +26.61 | +25.61 |
| byline | 76 | +29.01 | +28.70 | +27.63 |
| cross | 18 | +31.69 | +32.22 | +11.11 |
| no_span | 162 | +1.57 | +1.54 | +1.23 |

## Step 4 — Inspect the required candidate rank


In [4]:
# How deep must a reranker look to recover each example's best Top-20 ROUGE-L candidate?
records=[json.loads(line) for line in zf.read("topk_oracle_results/topk_candidates.jsonl").decode().splitlines()]
ranks=[]
for r in records:
    vals=[x["rougeL"] for x in r["candidates"]]; best=max(vals); ranks.append(next(i+1 for i,v in enumerate(vals) if v==best))
counts=collections.Counter(ranks); cumulative=0; lines=["| Rank needed | Examples whose Top-20 best first appears here | Cumulative |","|---:|---:|---:|"]
for rank in range(1,21):
    cumulative+=counts[rank]; lines.append(f"| {rank} | {counts[rank]} | {100*cumulative/len(ranks):.1f}% |")
display(Markdown("## Rank of the best available candidate\n\n"+"\n".join(lines)))


## Rank of the best available candidate

| Rank needed | Examples whose Top-20 best first appears here | Cumulative |
|---:|---:|---:|
| 1 | 520 | 48.3% |
| 2 | 104 | 58.0% |
| 3 | 62 | 63.8% |
| 4 | 48 | 68.2% |
| 5 | 38 | 71.7% |
| 6 | 43 | 75.7% |
| 7 | 29 | 78.4% |
| 8 | 27 | 80.9% |
| 9 | 19 | 82.7% |
| 10 | 23 | 84.9% |
| 11 | 18 | 86.5% |
| 12 | 21 | 88.5% |
| 13 | 15 | 89.9% |
| 14 | 16 | 91.4% |
| 15 | 12 | 92.5% |
| 16 | 5 | 92.9% |
| 17 | 20 | 94.8% |
| 18 | 13 | 96.0% |
| 19 | 22 | 98.0% |
| 20 | 21 | 100.0% |

## Step 5 — Write the conclusion


In [5]:
top1=summary["ks"]["1"]; top10=summary["ks"]["10"]; gain=top10["rougeL"]-top1["rougeL"]
decision="substantial" if gain>=3 else "limited" if gain<1 else "moderate"
text=f"""# Top-K span oracle analysis

For the current gated model, Top-1 ROUGE-L is **{top1['rougeL']:.2f}** and the Top-10 oracle ceiling is
**{top10['rougeL']:.2f}**, a **{gain:+.2f}** point gain. This indicates **{decision}** reranking headroom.
Top-10 exact-answer coverage is **{top10['exact_coverage']:.2f}%**. The oracle uses validation gold answers
only to measure candidate availability; it is not a deployable reranker. The locked test set was not accessed.
"""
Path("topk_span_oracle_report.md").write_text(text,encoding="utf-8"); display(Markdown("## Report-ready conclusion\n\n"+text)); zf.close()


## Report-ready conclusion

# Top-K span oracle analysis

For the current gated model, Top-1 ROUGE-L is **44.14** and the Top-10 oracle ceiling is
**67.16**, a **+23.02** point gain. This indicates **substantial** reranking headroom.
Top-10 exact-answer coverage is **49.16%**. The oracle uses validation gold answers
only to measure candidate availability; it is not a deployable reranker. The locked test set was not accessed.
